In [2]:
import torch
import torchvision
import torch.utils.data
import torchvision.transforms.v2

transforms = torchvision.transforms.v2.Compose(
    [
        torchvision.transforms.v2.Resize(224),
        torchvision.transforms.v2.Grayscale(num_output_channels=3),
        torchvision.transforms.v2.ToImage(),
        torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
        torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
    ]
)
train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)
classes = train_ds.class_to_idx

K_NEIGHBORS = 4
MAX_DIMENSION = 1

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import zigzag.nn
import zigzag.utils

dumper = zigzag.utils.UniversalDumper("mnist/vit_b_16/4_neighbors")
# dumper = zigzag.utils.UniversalDumper("mnist/vit_small")

model = torchvision.models.vit_b_16(
    num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT
)
model.heads = torch.nn.Identity()

train_embeddings = dumper.execute(
    zigzag.nn.precompute_embeddings,
    "train_embeddings",
    model,
    torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=False, num_workers=3)
)
test_embeddings = dumper.execute(
    zigzag.nn.precompute_embeddings,
    "test_embeddings",
    model,
    torch.utils.data.DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=3)
)

Got the result from mnist/vit_b_16/4_neighbors/train_embeddings.pt
Got the result from mnist/vit_b_16/4_neighbors/test_embeddings.pt


In [4]:
train_emb_ds = torch.utils.data.TensorDataset(train_embeddings, train_ds.targets)
test_emb_ds = torch.utils.data.TensorDataset(test_embeddings, test_ds.targets)

head = torch.nn.Sequential(torch.nn.LazyLinear(len(classes)), torch.nn.Softmax(dim=1))
train_history = dumper.execute(
    zigzag.nn.train,
    "train_history",
    head,
    torch.utils.data.DataLoader(train_emb_ds, batch_size=128, shuffle=True, num_workers=3),
    torch.utils.data.DataLoader(test_emb_ds, batch_size=128, shuffle=False, num_workers=3),
)

Got the result from mnist/vit_b_16/4_neighbors/train_history.csv


/usr/local/lib/python3.12/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [5]:
train_img_ds = train_ds

hidden_states = dumper.execute(
    zigzag.nn.collect_hidden_states,
    "hidden_states",
    model,
    torch.utils.data.DataLoader(train_img_ds, batch_size=128, shuffle=False, num_workers=3)
)
num_layers = len(hidden_states)

Got the result from mnist/vit_b_16/4_neighbors/hidden_states
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/0.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/1.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/2.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/3.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/4.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/5.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/6.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/7.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/8.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/9.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/10.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/11.pt
Got the result from mnist/vit_b_16/4_neighbors/hidden_states/12.pt


In [ ]:
import zigzag.topology

knn_graphs = dumper.execute(
    zigzag.topology.make_knn_graphs_vector,
    "knn_graphs",
    hidden_states,
    k_neighbors=K_NEIGHBORS
)
diagrams = dumper.execute(
    zigzag.topology.compute_zigzag_barcodes,
    "diagrams",
    knn_graphs,
    dimension=MAX_DIMENSION
)

Got the result from mnist/vit_b_16/4_neighbors/knn_graphs
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/0.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/1.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/2.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/3.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/4.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/5.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/6.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/7.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/8.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/9.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/10.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/11.npz
Got the result from mnist/vit_b_16/4_neighbors/knn_graphs/12.npz


Generate simplex tree: 100%|██████████| 13/13 [04:14<00:00, 19.61s/it]


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize = (13, 7))
for diagram, ax in zip(diagrams, axes.flat):
    zigzag.topology.plot_persistence_image(diagram, num_layers, ax=ax)
fig.savefig(f"{dumper.directory_}/persistence_image.png")
fig.savefig(f"{dumper.directory_}/persistence_image.svg")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (13, 7))
for diagram, ax in zip(diagrams, axes.flat):
    zigzag.topology.plot_weighted_inter_layer_persistence(diagram, num_layers, ax=ax)
fig.savefig(f"{dumper.directory_}/inter_layer_persistence.png")
fig.savefig(f"{dumper.directory_}/inter_layer_persistence.svg")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (13, 7))
for diagram, ax in zip(diagrams, axes.flat):
    zigzag.topology.plot_births_relative_frequency(diagram, num_layers, ax=ax)
fig.savefig(f"{dumper.directory_}/births_relative_frequency.png")
fig.savefig(f"{dumper.directory_}/births_relative_frequency.svg")